# Functions

## Function for figure 1

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

def draw_equal_quincunx_with_arrows(radius=1.0, arrow_len=None, savepath=None):
    """
    Five equal circles (radius = radius) in quincunx.
    Outer four are tangent to the center circle (centers at distance d=2*radius).
    Dashed guide circle goes through the outer centers.
    Arrows start at the CENTER of each outer circle:
        top & bottom: upwards; left & right: downwards.
    The dashed guide is rendered BEHIND the disks (hidden where overlapped).
    """
    s = float(radius)
    if s <= 0:
        raise ValueError("radius must be positive.")

    d = 2.0 * s
    if arrow_len is None:
        arrow_len = 1.1 * s

    centers = {
        "top":    (0.0,  d),
        "right":  ( d,  0.0),
        "bottom": (0.0, -d),
        "left":   (-d,  0.0),
        "center": (0.0,  0.0),
    }
    # z-order: guide (1) < circles (3) < arrows (4)
    z_guide, z_circles, z_arrows = 1, 3, 4
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    ax.axis('off')
    # --- dashed guide circle FIRST (behind) ---
    ax.add_patch(Circle((0.0, 0.0), d, ec="#4a5568", fc="none", lw=1.8, ls="--", zorder=z_guide))
    # --- disks on top of guide (opaque facecolors hide the dashed where overlapped) ---
    for key in ["top", "right", "bottom", "left", "center"]:
        cx, cy = centers[key]
        ax.add_patch(Circle((cx, cy), s, ec="#2b6cb0", fc="#e6eefc", lw=2, zorder=z_circles))
    # --- arrows on top ---
    def vertical_arrow_from_center(cx, cy, direction):
        if direction == "up":
            start, end = (cx, cy), (cx, cy + arrow_len)
        elif direction == "down":
            start, end = (cx, cy), (cx, cy - arrow_len)
        else:
            raise ValueError("direction must be 'up' or 'down'")
        a = FancyArrowPatch(
            start, end, arrowstyle='-|>', mutation_scale=16,
            lw=2.0, color='#44515c', zorder=z_arrows
        )
        ax.add_patch(a)
    vertical_arrow_from_center(*centers["top"],    "up")
    vertical_arrow_from_center(*centers["bottom"], "up")
    vertical_arrow_from_center(*centers["left"],   "down")
    vertical_arrow_from_center(*centers["right"],  "down")
    # limits include arrows
    pad = 0.3 * s
    x_pad = s + pad
    y_pad = max(s, arrow_len) + pad
    ax.set_xlim(-d - x_pad, d + x_pad)
    ax.set_ylim(-d - y_pad, d + y_pad)
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()
    plt.close(fig)

## Function for figure 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrow, Arc

def draw_rolling_diagram(R=3, r=1, theta_deg=30, save_path=None):
    """
    Rolling-circle diagram with:
      - v drawn from the small-circle center (longer, tangent to center path),
      - ω curved arrow placed on TOP of the small circle (outside) to avoid crossings,
      - R and r shown as teal radius segments with midpoint labels.
    """
    theta = np.deg2rad(theta_deg)
    RR = R + r
    O = (0.0, 0.0)
    C = (RR*np.cos(theta), RR*np.sin(theta))
    fig, ax = plt.subplots(figsize=(6, 6))
    col, teal = '#1f2d3a', '#1f6f7a'
    # Dashed path of the rolling center
    ax.add_patch(Circle(O, RR, fill=False, ls='--', lw=1.5, color=col))
    # Circles
    ax.add_patch(Circle(O, R, fill=False, lw=2, color=col))
    ax.add_patch(Circle(C, r, fill=False, lw=2, color=col))
    # Direction from O to C
    ang = np.arctan2(C[1]-O[1], C[0]-O[0])
    # Radius R with midpoint label
    R_end = (O[0] + R*np.cos(ang), O[1] + R*np.sin(ang))
    ax.plot([O[0], R_end[0]], [O[1], R_end[1]], color='#1f6f7a', lw=2)
    R_mid = (O[0] + 0.5*R*np.cos(ang), O[1] + 0.5*R*np.sin(ang))
    ax.text(R_mid[0], R_mid[1], r"$R$", color='#1f6f7a', fontsize=12, ha='center', va='bottom')
    # Radius r with midpoint label
    r_end = (C[0] - r*np.cos(ang), C[1] - r*np.sin(ang))
    ax.plot([C[0], r_end[0]], [C[1], r_end[1]], color='#1f6f7a', lw=2)
    r_mid = (C[0] - 0.5*r*np.cos(ang), C[1] - 0.5*r*np.sin(ang))
    ax.text(r_mid[0], r_mid[1], r"$r$", color='#1f6f7a', fontsize=12, ha='center', va='bottom')
    # Velocity vector from small-circle center (tangent)
    vx, vy = np.cos(ang + np.pi/2), np.sin(ang + np.pi/2)
    v_len = 1.8
    ax.add_patch(FancyArrow(C[0], C[1], v_len*vx, v_len*vy,width=0.025, head_width=0.22, head_length=0.22, color='#1f6f7a'))
    ax.text(C[0] + v_len*vx + 0.12, C[1] + v_len*vy, r"$\vec{v}$", color='#1f6f7a', fontsize=12)
    # ω curved arrow OUTSIDE small circle, on top (absolute angles 70°→110°)
    arc_rad = 1.20 * r
    start_deg_abs, end_deg_abs = 70, 110
    arc = Arc(C, 2*arc_rad, 2*arc_rad, theta1=start_deg_abs, theta2=end_deg_abs, color='#1f6f7a', lw=2)
    ax.add_patch(arc)
    end_rad = np.deg2rad(end_deg_abs)
    hx = C[0] + arc_rad*np.cos(end_rad)
    hy = C[1] + arc_rad*np.sin(end_rad)
    tdx, tdy = -np.sin(end_rad), np.cos(end_rad)  # CCW tangent
    ax.add_patch(FancyArrow(hx-0.12*tdx, hy-0.12*tdy,0.12*tdx, 0.12*tdy,width=0.0, head_width=0.18, head_length=0.18,
                            color='#1f6f7a', length_includes_head=True))
    ax.text(hx + 0.22, hy + 0.18, r"$\omega$", color='#1f6f7a', fontsize=12)
    # Layout
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-RR-1.4, RR+1.8)
    ax.set_ylim(-RR-1.4, RR+1.8)
    ax.axis('off')
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    return fig, ax


## Function for figure 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

def plot_epicycloid(R=3.0, r=1.0, points=2000, save_path=None, show=True):
    """
    Plot the epicycloid generated by a circle of radius r rolling
    around a fixed circle of radius R.

    Parameters
    ----------
    R : float
        Radius of the fixed circle.
    r : float
        Radius of the rolling circle.
    points : int
        Number of sample points for the epicycloid.
    save_path : str or None
        If provided, saves the figure to this path.
    show : bool
        If True, displays the figure. If False, closes it.
    
    Returns
    -------
    fig, ax : Matplotlib figure and axes objects
    """
    k = (R + r) / r
    theta = np.linspace(0, 2*np.pi, points)
    x = (R + r) * np.cos(theta) - r * np.cos(k * theta)
    y = (R + r) * np.sin(theta) - r * np.sin(k * theta)
    fig, ax = plt.subplots(figsize=(6,6))
    # Grid
    ax.set_xlim(-R-r-1, R+r+1)
    ax.set_ylim(-R-r-1, R+r+1)
    ax.set_aspect('equal', 'box')
    ax.set_xticks(np.arange(-R-r, R+r+1, 1))
    ax.set_yticks(np.arange(-R-r, R+r+1, 1))
    ax.grid(True, which='both', linestyle='-', linewidth=0.5, alpha=0.25)
    # Axes lines
    ax.axhline(0, color='k', linewidth=1)
    ax.axvline(0, color='k', linewidth=1)
    ax.text(R+r-0.5, -0.3, 'x', fontsize=12)
    ax.text(0.2, R+r-0.5, 'y', fontsize=12)
    # Fixed circle (blue)
    ax.add_patch(Circle((0,0), R, fill=False, edgecolor='blue', linewidth=2))
    # Rolling circle (black) at initial position
    rolling_center = (R + r, 0.0)
    ax.add_patch(Circle(rolling_center, r, fill=False, edgecolor='black', linewidth=2))
    # Center dot
    ax.plot(rolling_center[0], rolling_center[1], 'ko')
    # Epicycloid path
    ax.plot(x, y, color='red', linewidth=2)
    # Starting point
    ax.plot([R], [0], 'o', color='red')
    # Clean frame
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    if not show:
        plt.close(fig)
    return fig, ax


## Function for figure 3 animated

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

rc("animation", html="jshtml")   # for Jupyter animation display

def animate_epicycloid(R=3.0, r=1.0, frames=600, interval=20):
    """
    Animate the epicycloid generated by a circle of radius r rolling
    around a fixed circle of radius R.

    Parameters
    ----------
    R : float
        Radius of the fixed circle.
    r : float
        Radius of the rolling circle.
    frames : int
        Number of frames in the animation.
    interval : int
        Delay between frames in milliseconds.

    Returns
    -------
    ani : matplotlib.animation.FuncAnimation
        Animation object (renders inline in Jupyter).
    """
    k = (R + r) / r
    theta = np.linspace(0, 2*np.pi, frames)
    x = (R + r) * np.cos(theta) - r * np.cos(k * theta)
    y = (R + r) * np.sin(theta) - r * np.sin(k * theta)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.set_xlim(-R-r-1, R+r+1)
    ax.set_ylim(-R-r-1, R+r+1)
    ax.set_aspect('equal', 'box')
    ax.set_xticks(np.arange(-R-r, R+r+1, 1))
    ax.set_yticks(np.arange(-R-r, R+r+1, 1))
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.25)
    ax.axhline(0, color='k', linewidth=1)
    ax.axvline(0, color='k', linewidth=1)
    ax.text(R+r-0.5, -0.3, 'x')
    ax.text(0.2, R+r-0.5, 'y')
    for s in ax.spines.values(): 
        s.set_visible(False)
    # Fixed circle
    ax.add_patch(Circle((0,0), R, fill=False, edgecolor='blue', linewidth=2))
    # Rolling circle
    rolling_circle = Circle((R+r, 0), r, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rolling_circle)
    # Traced point and path
    trace_dot, = ax.plot([], [], 'ro')
    trace_line, = ax.plot([], [], 'r-', linewidth=2)
    # Starting point
    ax.plot([R], [0], 'o', color='red')
    plt.close(fig)
    def init():
        rolling_circle.center = (R+r, 0)
        trace_dot.set_data([], [])
        trace_line.set_data([], [])
        return trace_dot, trace_line, rolling_circle
    def update(i):
        cx = (R + r) * np.cos(theta[i])
        cy = (R + r) * np.sin(theta[i])
        rolling_circle.center = (cx, cy)
        trace_dot.set_data(x[i], y[i])
        trace_line.set_data(x[:i+1], y[:i+1])
        return trace_dot, trace_line, rolling_circle
    ani = FuncAnimation(fig, update, init_func=init, frames=frames, interval=interval, blit=True)
    return ani


## Function for figure 5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_epicycloid_panel(
    k_list,
    subtitles=None,
    r=1.0,
    num_points=4000,
    figsize=(22, 12),
    savepath="fig5.png",
    dpi=300
):
    # Internal function to compute epicycloid coordinates
    def epicycloid_xy(k, turns):
        R = k * r
        w = (R + r) / r
        t = np.linspace(0, 2 * np.pi * turns, num_points)
        x = (R + r) * np.cos(t) - r * np.cos(w * t)
        y = (R + r) * np.sin(t) - r * np.sin(w * t)
        return x, y

    # Internal function to draw one panel
    def plot_panel(ax, k, subtitle=None):
        turns = 1.0 if abs(k - round(k)) < 1e-9 else 12.0
        x, y = epicycloid_xy(k, turns)
        ax.axhline(0, color='0.6', lw=0.8, ls='--')
        ax.axvline(0, color='0.6', lw=0.8, ls='--')
        ax.plot(x, y, color='black', lw=1.8)
        ax.set_aspect('equal', 'box')
        lim = k + 3.0
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(1.0)
            spine.set_color('0.3')
        if subtitle:
            ax.text(
                0.5, -0.10, subtitle,
                transform=ax.transAxes,
                ha='center', va='top',
                fontsize=22
            )
    # Infer grid size automatically
    n = len(k_list)
    nrows = int(np.ceil(n / 4))
    ncols = min(4, n)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for i, k in enumerate(k_list):
        subtitle = subtitles[i] if subtitles else None
        plot_panel(axes[i], k, subtitle)
    # Remove unused axes if any
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    plt.subplots_adjust(
        left=0.00, right=1.0,
        top=1.0, bottom=0.1,
        wspace=0.1, hspace=0.3
    )
    if savepath:
        plt.savefig(savepath, dpi=dpi, bbox_inches="tight")
    plt.show()

## Function for figure 6

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrow
import numpy as np

def draw_coin(ax, center, radius, facecolor='#87CEEB', edgecolor='black',
              arm_angle=None, arm_len=None, armcolor='black', lw=3):
    """Draw a coin with an optional orientation arrow."""
    circ = Circle(center, radius, facecolor=facecolor, edgecolor=edgecolor, lw=lw)
    ax.add_patch(circ)
    # center dot
    ax.plot(center[0], center[1], 'o', color='black', ms=4)
    # orientation arrow
    if arm_angle is not None:
        if arm_len is None:
            arm_len = radius * 0.9
        x0, y0 = center
        x1 = x0 + arm_len * np.cos(arm_angle)
        y1 = y0 + arm_len * np.sin(arm_angle)
        arr = FancyArrow(x0, y0, x1 - x0, y1 - y0, width=0.03*radius, head_width=0.25*radius, head_length=0.25*radius,
                         color=armcolor, length_includes_head=True)
        ax.add_patch(arr)

def tangent_center(big_c, R, r, angle):
    """Center of a small circle tangent to the big circle at polar angle 'angle'."""
    return (big_c[0] + (R + r) * np.cos(angle),
            big_c[1] + (R + r) * np.sin(angle))

def draw_scene(ax, big_center, R, r, mode='left',
               big_color='#87CEEB', small_color='#B0E0E6'):
    """Draw one panel (left or right)."""
    # Big coin
    draw_coin(ax, big_center, R, facecolor=big_color, lw=3)
    # Top small coin (tangent at 90°) with downward arrow
    top_c = tangent_center(big_center, R, r, np.pi/2)
    draw_coin(ax, top_c, r, facecolor=big_color, lw=3, arm_angle=-np.pi/2)
    if mode == 'left':
        # Two tangent small coins at 210° and 330° with angled arrows
        c1 = tangent_center(big_center, R, r, np.deg2rad(210))
        c2 = tangent_center(big_center, R, r, np.deg2rad(330))
        draw_coin(ax, c1, r, facecolor=small_color, edgecolor='#6b6b6b', arm_angle=np.deg2rad(30))
        draw_coin(ax, c2, r, facecolor=small_color, edgecolor='#6b6b6b', arm_angle=np.deg2rad(150))
    else:
        # Three tangent coins: left (180°), right (0°), bottom (270°)
        left_c   = tangent_center(big_center, R, r, np.pi)
        right_c  = tangent_center(big_center, R, r, 0.0)
        bottom_c = tangent_center(big_center, R, r, -np.pi/2)
        # Left & right coins: arrows up
        draw_coin(ax, left_c,  r, facecolor=small_color, edgecolor='#6b6b6b', arm_angle=-np.pi/2)
        draw_coin(ax, right_c, r, facecolor=small_color, edgecolor='#6b6b6b', arm_angle=-np.pi/2)
        # Bottom coin: arrow DOWN (fixed)
        draw_coin(ax, bottom_c, r, facecolor=small_color, edgecolor='#6b6b6b', arm_angle=-np.pi/2)

def print_image(r=0.6, k=3, center_distance=8.0, save_path=None):
    """Render the two panels and optionally save to file."""
    R = k * r  # enforce R = k r
    fig, ax = plt.subplots(figsize=(10, 5))
    left_center  = (-center_distance/2, 0.0)
    right_center = ( center_distance/2, 0.0)
    draw_scene(ax, left_center,  R, r, mode='left')
    draw_scene(ax, right_center, R, r, mode='right')
    # Labels (a) and (b) under each panel
    y_label = - (R + r) - 0.8
    ax.text(left_center[0],  y_label, '(a)', fontsize=14, ha='center', va='top')
    ax.text(right_center[0], y_label, '(b)', fontsize=14, ha='center', va='top')
    # Layout
    extent = center_distance/2 + (R + r) + 1.0
    ax.set_xlim(-extent, extent)
    ax.set_ylim(- (R + r) - 1.5, (R + r) + 1.5)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

## Function for figure 7

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

def plot_hypocycloid(R=3.0, r=1.0, points=2000, save_path=None, show=True):
    """
    Plot the epicycloid generated by a circle of radius r rolling
    around a fixed circle of radius R.

    Parameters
    ----------
    R : float
        Radius of the fixed circle.
    r : float
        Radius of the rolling circle.
    points : int
        Number of sample points for the epicycloid.
    save_path : str or None
        If provided, saves the figure to this path.
    show : bool
        If True, displays the figure. If False, closes it.
    
    Returns
    -------
    fig, ax : Matplotlib figure and axes objects
    """
    k = -(R - r) / r
    theta = np.linspace(0, 2*np.pi, points)
    x = (R - r) * np.cos(theta) + r * np.cos(k * theta)
    y = (R - r) * np.sin(theta) + r * np.sin(k * theta)
    fig, ax = plt.subplots(figsize=(6,6))
    # Grid
    ax.set_xlim(-R-r-1, R+r+1)
    ax.set_ylim(-R-r-1, R+r+1)
    ax.set_aspect('equal', 'box')
    ax.set_xticks(np.arange(-R-r, R+r+1, 1))
    ax.set_yticks(np.arange(-R-r, R+r+1, 1))
    ax.grid(True, which='both', linestyle='-', linewidth=0.5, alpha=0.25)
    # Axes lines
    ax.axhline(0, color='k', linewidth=1)
    ax.axvline(0, color='k', linewidth=1)
    ax.text(R+r-0.5, -0.3, 'x', fontsize=12)
    ax.text(0.2, R+r-0.5, 'y', fontsize=12)
    # Fixed circle (blue)
    ax.add_patch(Circle((0,0), R, fill=False, edgecolor='blue', linewidth=2))
    # Rolling circle (black) at initial position
    rolling_center = (R - r, 0.0)
    ax.add_patch(Circle(rolling_center, r, fill=False, edgecolor='black', linewidth=2))
    # Center dot
    ax.plot(rolling_center[0], rolling_center[1], 'ko')
    # Epicycloid path
    ax.plot(x, y, color='red', linewidth=2)
    # Starting point
    ax.plot([R], [0], 'o', color='red')
    # Clean frame
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    if not show:
        plt.close(fig)
    return fig, ax


## Function for figure 7 animated

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

rc("animation", html="jshtml")   # for Jupyter animation display

def animate_hypocycloid(R=3.0, r=1.0, frames=600, interval=20):
    """
    Animate the epicycloid generated by a circle of radius r rolling
    around a fixed circle of radius R.
    Parameters
    ----------
    R : float
        Radius of the fixed circle.
    r : float
        Radius of the rolling circle.
    frames : int
        Number of frames in the animation.
    interval : int
        Delay between frames in milliseconds.

    Returns
    -------
    ani : matplotlib.animation.FuncAnimation
        Animation object (renders inline in Jupyter).
    """
    k = - (R - r) / r
    theta = np.linspace(0, 2*np.pi, frames)
    x = (R - r) * np.cos(theta) + r * np.cos(k * theta)
    y = (R - r) * np.sin(theta) + r * np.sin(k * theta)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.set_xlim(-R-r-1, R+r+1)
    ax.set_ylim(-R-r-1, R+r+1)
    ax.set_aspect('equal', 'box')
    ax.set_xticks(np.arange(-R-r, R+r+1, 1))
    ax.set_yticks(np.arange(-R-r, R+r+1, 1))
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.25)
    ax.axhline(0, color='k', linewidth=1)
    ax.axvline(0, color='k', linewidth=1)
    ax.text(R+r-0.5, -0.3, 'x')
    ax.text(0.2, R+r-0.5, 'y')
    for s in ax.spines.values(): 
        s.set_visible(False)
    # Fixed circle
    ax.add_patch(Circle((0,0), R, fill=False, edgecolor='blue', linewidth=2))
    # Rolling circle
    rolling_circle = Circle((R+r, 0), r, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rolling_circle)
    # Traced point and path
    trace_dot, = ax.plot([], [], 'ro')
    trace_line, = ax.plot([], [], 'r-', linewidth=2)
    # Starting point
    ax.plot([R], [0], 'o', color='red')
    plt.close(fig)
    def init():
        rolling_circle.center = (R+r, 0)
        trace_dot.set_data([], [])
        trace_line.set_data([], [])
        return trace_dot, trace_line, rolling_circle
    def update(i):
        cx = (R - r) * np.cos(theta[i])
        cy = (R - r) * np.sin(theta[i])
        rolling_circle.center = (cx, cy)
        trace_dot.set_data(x[i], y[i])
        trace_line.set_data(x[:i+1], y[:i+1])
        return trace_dot, trace_line, rolling_circle
    ani = FuncAnimation(fig, update, init_func=init, frames=frames, interval=interval, blit=True)
    return ani

## Function for figure 8

In [ ]:
# Import NumPy for numerical computations and array generation
import numpy as np
# Import Matplotlib pyplot for plotting and figure handling
import matplotlib.pyplot as plt

# Define a function that generates hypocycloid (epicycloid-style) coordinates
def plot_hypocycloid_panel(
    # List of k values controlling the radius ratio
    k_list,
    # Optional list of subtitles for each panel
    subtitles=None,
    # Radius of the rolling circle
    r=1.0,
    # Number of points used to draw each curve
    num_points=4000,
    # Size of the full figure
    figsize=(22, 12),
    # Output filename for saving the figure
    savepath="fig8.png",
    # Resolution of the saved figure
    dpi=300
):
    # Define an internal function to compute hypocycloid coordinates
    def hypocycloid_xy(k, turns):
        # Compute the radius of the fixed circle
        R = k * r
        # Compute the angular frequency with negative sign for rolling direction
        w = - (R - r) / r
        # Create an array of parameter values
        t = np.linspace(0, 2 * np.pi * turns, num_points)
        # Compute the x-coordinates of the hypocycloid
        x = (R - r) * np.cos(t) + r * np.cos(w * t)
        # Compute the y-coordinates of the hypocycloid
        y = (R - r) * np.sin(t) + r * np.sin(w * t)
        # Return the parametric coordinates
        return x, y

    # Define an internal function to draw a single subplot panel
    def plot_panel(ax, k, subtitle=None):
        # Choose number of turns depending on whether k is integer
        turns = 1.0 if abs(k - round(k)) < 1e-9 else 12.0
        # Generate the curve coordinates
        x, y = hypocycloid_xy(k, turns)
        # Draw a horizontal reference line through the origin
        ax.axhline(0, color='0.6', lw=0.8, ls='--')
        # Draw a vertical reference line through the origin
        ax.axvline(0, color='0.6', lw=0.8, ls='--')
        # Plot the hypocycloid curve
        ax.plot(x, y, color='black', lw=1.8)
        # Enforce equal scaling on both axes
        ax.set_aspect('equal', 'box')
        # Define plot limits based on k
        lim = k + 3.0
        # Set x-axis limits
        ax.set_xlim(-lim, lim)
        # Set y-axis limits
        ax.set_ylim(-lim, lim)
        # Remove x-axis tick marks
        ax.set_xticks([])
        # Remove y-axis tick marks
        ax.set_yticks([])
        # Iterate over all axis spines
        for spine in ax.spines.values():
            # Set spine line width
            spine.set_linewidth(1.0)
            # Set spine color
            spine.set_color('0.3')
        # Add subtitle text below the panel if provided
        if subtitle:
            ax.text(
                0.5, -0.10, subtitle,
                transform=ax.transAxes,
                ha='center', va='top',
                fontsize=22
            )

    # Determine total number of panels
    n = len(k_list)
    # Fix the number of columns to four
    ncols = 4
    # Compute the required number of rows
    nrows = int(np.ceil(n / ncols))

    # Create the figure and subplot grid
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    # Flatten the axes array for easy indexing
    axes = np.atleast_1d(axes).ravel()

    # Loop over all k values and corresponding axes
    for i, k in enumerate(k_list):
        # Select the corresponding subtitle if available
        subtitle = subtitles[i] if subtitles else None
        # Plot the panel for the current k value
        plot_panel(axes[i], k, subtitle)

    # Remove any unused subplot axes
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # Adjust subplot spacing to maximize panel size
    plt.subplots_adjust(
        left=0.00,
        right=1.0,
        top=1.0,
        bottom=0.1,
        wspace=0.1,
        hspace=0.3
    )

    # Save the figure to disk if a path is provided
    if savepath:
        plt.savefig(savepath, dpi=dpi, bbox_inches="tight")

    # Display the figure on screen
    plt.show()

## Function for figure 9

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_aristotle_wheel(R=2.0, r=1.0, savepath=None):
    """
    Plot Aristotle's Wheel Paradox diagram for two concentric rolling circles.

    Parameters
    ----------
    R : float
        Radius of the larger circle.
    r : float
        Radius of the smaller circle.
    savepath : str or None
        If provided, saves the figure to the given path.
    """
    theta = np.linspace(0, 2 * np.pi, 600)

    # General trochoid for a point at distance a from center
    def trochoid(a):
        x = R * theta + a * np.sin(theta)
        y = R + a * np.cos(theta)
        return x, y

    # Trajectories for P_R (a=R) and P_r (a=r)
    x_R, y_R = trochoid(R)
    x_r, y_r = trochoid(r)

    # Center line at y = R, spanning exactly 2πR
    x_line = [0.0, 2 * np.pi * R]
    y_line = [R, R]

    # Figure/axes
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_aspect('equal')
    ax.axis('off')

    # Paths
    ax.plot(x_R, y_R, 'b--', lw=1.5, label="Path of $P_R$")
    ax.plot(x_r, y_r, 'r--', lw=1.5, label="Path of $P_r$")
    ax.plot(x_line, y_line, 'g--', lw=1.2, label="Center line")

    # Vertical black lines at start and end (center to top of large circle)
    ax.plot([0, 0], [R, R + R], 'k', lw=1)
    ax.plot([2 * np.pi * R, 2 * np.pi * R], [R, R + R], 'k', lw=1)

    # Circles at start (x=0) and end (x=2πR)
    left_large  = plt.Circle((0.0, R), R, edgecolor='blue', fill=False, lw=1.5)
    left_small  = plt.Circle((0.0, R), r, edgecolor='red',  fill=False, lw=1.5)
    right_large = plt.Circle((2 * np.pi * R, R), R, edgecolor='blue', fill=False, lw=1.5)
    right_small = plt.Circle((2 * np.pi * R, R), r, edgecolor='red',  fill=False, lw=1.5)
    for c in (left_large, left_small, right_large, right_small):
        ax.add_patch(c)

    # Markers and labels: top points on each circle at start/end
    PR_left  = (0.0, R + R)
    Pr_left  = (0.0, R + r)
    PR_right = (2 * np.pi * R, R + R)
    Pr_right = (2 * np.pi * R, R + r)

    ax.plot(*PR_left,  'bo', markersize=6, mfc='white')
    ax.plot(*Pr_left,  'ro', markersize=6, mfc='white')
    ax.plot(*PR_right, 'bo', markersize=6, mfc='white')
    ax.plot(*Pr_right, 'ro', markersize=6, mfc='white')

    ax.text(PR_left[0]  - 0.2, PR_left[1]  + 0.2, r"$P_R$", fontsize=12, color='blue')
    ax.text(Pr_left[0]  - 0.2, Pr_left[1]  + 0.2, r"$P_r$", fontsize=12, color='red')
    ax.text(PR_right[0] + 0.1, PR_right[1] + 0.2, r"$P_R$", fontsize=12, color='blue')
    ax.text(Pr_right[0] + 0.1, Pr_right[1] + 0.2, r"$P_r$", fontsize=12, color='red')

    # Ground line
    x_pad = max(R, r) * 0.8
    ax.plot([-x_pad, 2 * np.pi * R + x_pad], [0, 0], color='peru', lw=3)

    # Robust limits: include full circles and labels for any R, r
    pad_y = max(R, r) * 0.3
    y_min = min(0.0, R - max(R, r)) - pad_y
    y_max = R + max(R, r) + pad_y
    x_min = -R - x_pad
    x_max = 2 * np.pi * R + R + x_pad

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches='tight', pad_inches=0.1)

    plt.close(fig)  # close to avoid handle leaks
    return fig

## Function for figure 9 animated

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_aristotle_wheel(R=2.0, r=1.0, frames=300, interval=30, show_trails=True):
    """
    Animate Aristotle's Wheel Paradox: two concentric circles of radii R and r
    roll without slipping over a distance 2πR. Points P_R (a=R) and P_r (a=r)
    trace trochoids. Returns HTML for inline display in Jupyter.
    """
    # Parameter over one full revolution
    theta = np.linspace(0.0, 2*np.pi, frames)

    # Helpers
    def trochoid(a, th):
        x = R*th + a*np.sin(th)
        y = R   + a*np.cos(th)
        return x, y

    # Precompute paths for trails
    xR_path, yR_path = trochoid(R, theta)   # P_R path
    xr_path, yr_path = trochoid(r, theta)   # P_r path

    # Figure and axes
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_aspect('equal')
    ax.axis('off')

    # Static floor and center line
    x_total = 2*np.pi*R
    pad_x = max(R, r)*0.8
    ax.plot([-pad_x, x_total + pad_x], [0, 0], color='peru', lw=3)   # floor
    ax.plot([0, x_total], [R, R], 'g--', lw=1.2)                     # center line y=R

    # Static end markers: vertical lines at start and end
    ax.plot([0, 0], [R, R+R], 'k', lw=1)
    ax.plot([x_total, x_total], [R, R+R], 'k', lw=1)

    # Dynamic patches: rolling concentric circles (share same center)
    big = Circle((0, R), R, fill=False, lw=1.8, ec='blue')
    small = Circle((0, R), r, fill=False, lw=1.8, ec='red')
    ax.add_patch(big)
    ax.add_patch(small)

    # Moving points P_R and P_r
    PR_dot, = ax.plot([], [], 'o', ms=6, mfc='white', mec='blue')
    Pr_dot, = ax.plot([], [], 'o', ms=6, mfc='white', mec='red')

    # Labels near points
    PR_lbl = ax.text(0, 0, r"$P_R$", fontsize=12, color='blue',
                     ha='left', va='bottom')
    Pr_lbl = ax.text(0, 0, r"$P_r$", fontsize=12, color='red',
                     ha='left', va='bottom')

    # Optional trails
    PR_trail, = ax.plot([], [], 'b--', lw=1.2) if show_trails else (None,)
    Pr_trail, = ax.plot([], [], 'r--', lw=1.2) if show_trails else (None,)

    # Robust limits (avoid clipping for any R, r)
    pad_y = max(R, r)*0.35
    y_min = min(0.0, R - max(R, r)) - pad_y
    y_max = R + max(R, r) + pad_y
    x_min = -R - pad_x
    x_max = x_total + R + pad_x
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    # Animation update
    def update(i):
        th = theta[i]
        # Rolling center (translation only; rotation is encoded in trochoid points)
        center_x = R*th
        center_y = R
        big.center = (center_x, center_y)
        small.center = (center_x, center_y)

        # Current P_R and P_r
        xR, yR = trochoid(R, th)
        xr, yr = trochoid(r, th)

        PR_dot.set_data(xR, yR)
        Pr_dot.set_data(xr, yr)

        PR_lbl.set_position((xR + 0.08*R, yR + 0.08*R))
        Pr_lbl.set_position((xr + 0.08*R, yr + 0.08*R))

        # Update trails
        artists = [big, small, PR_dot, Pr_dot, PR_lbl, Pr_lbl]
        if show_trails:
            PR_trail.set_data(xR_path[:i+1], yR_path[:i+1])
            Pr_trail.set_data(xr_path[:i+1], yr_path[:i+1])
            artists += [PR_trail, Pr_trail]
        return artists

    anim = FuncAnimation(fig, update, frames=frames, interval=interval, blit=True)
    plt.close(fig)  # prevent duplicate static output
    return anim

# Figures

## Figure 1

![Image](./figures/fig1.png)

In [ ]:
# Example:
draw_equal_quincunx_with_arrows(radius=1.0, arrow_len=1.1, savepath='fig1.png')

## Figure 2

In [ ]:
# Example usage:
fig, ax = draw_rolling_diagram(R=3, r=1, theta_deg=30, save_path="fig2.png")

## Figure 3

In [ ]:
fig, ax = plot_epicycloid(R=3.0, r=1.0, save_path="fig3.png")

In [ ]:
ani = animate_epicycloid(R=3.0, r=1.0, frames=900, interval=30)
# Save to MP4 (requires ffmpeg installed)
ani.save("epycycloid.mp4", writer="ffmpeg", dpi=150)
ani  # shows animation

## Figure 4

![Image](./figures/fig4.png)

## Figure 5

In [ ]:
k_list = [1, 2, 3, 4, 2.5, 5.8, 25, 100]
subs = ["k = 1; cardioid", "k = 2; nephroid", "k = 3; trefoiloid", "k = 4; quatrefoiloid", 
        "k = 2.5", "k = 5.8", "k = 25", "k = 100"]
#
plot_epicycloid_panel(k_list, subtitles=subs)

## Figure 6

In [ ]:
# Example
print_image(r=0.6, k=3, center_distance=8.0, save_path="fig6.png")

## Figure 7

In [ ]:
fig, ax = plot_hypocycloid(R=3.0, r=1.0, save_path="fig7.png")

## Figure 7 animated

In [ ]:
ani = animate_hypocycloid(R=3.0, r=1.0, frames=600, interval=30)
# Save to MP4 (requires ffmpeg installed)
ani.save("hypocycloid.mp4", writer="ffmpeg", dpi=150)
ani  # shows animation

## Figure 8

In [ ]:
# Define the list of k values to be plotted
k_list = [3, 4, 5, 6, 2.1, 3.8, 5.5, 100]
# Define the subtitle corresponding to each k value
subtitles = ["k = 3; deltoid", "k = 4; astroid", "k = 5", "k = 6", "k = 2.1", "k = 3.8", "k = 5.5", "k = 100"]
# Call the function to generate and display the hypocycloid panels
plot_hypocycloid_panel(k_list=k_list, subtitles=subtitles, figsize=(22, 12), savepath="fig8.png", dpi=300)

## Figure 9

In [ ]:
# Save as PNG
fig = plot_aristotle_wheel(R=2, r=1, savepath="fig9.png")  
fig

## Figure 9 animated

In [ ]:
# Example (in a Jupyter cell):
anim = animate_aristotle_wheel(R=4.0, r=1.0)
# Save to MP4 (requires ffmpeg installed)
# anim.save("aristotle_wheel.mp4", writer="ffmpeg", dpi=150)
# Or save to GIF (requires ImageMagick or Pillow support)
# anim.save("aristotle_wheel.gif", writer="pillow", dpi=100)
# Display animation
HTML(anim.to_jshtml())

# References

[01] J.C. Maxwell, *XXXV on Trans. Roy. Soc. Edin.* **16**, 519 (1849).  
[02] E.H. Lockwood, in: *A Book of Curves* (Cambridge University Press, Cambridge, 1967).  
[03] H. Cundy and A. Rollett, in: *Mathematical Models* (Tarquin Publications, Stradbroke, 1989), 3 ed.  
[04] M. Gardner, *The Sixth Book of Mathematical Games from Scientific American* (University of Chicago Press, Chicago, 1984).  
[05] R.C. Yates, *A Handbook on Curves and Their Properties* (J.W. Edwards, Ann Arbor, 1952).  
[06] J.D. Lawrence, *A Catalog of Special Plane Curves* (Dover, New York, 1972).  
[07] D. Zwillinger, in: *CRC Standard Mathematical Tables and Formulae* (CRC Press, Boca Raton, 1996), 3 ed.  
[08] M. Gardner, in: *Mathematical Carnival* (Alfred A. Knopf, New York, 1975).  
[09] T. Pappas, in: *The Joy of Mathematics* (World Public./Tetra, San Carlos, 1989).  
[10] H. Steinhaus, *Mathematical Snapshots* (Dover, New York, 1999), 3 ed.  
[11] I.E. Drabkin, *Aristotle’s Wheel: Notes on the History of a Paradox* **9**, 162 (1950).  
[12] D.W. Ballew, *Math. Teach.* **65**, 507 (1972).  
[13] P. Costabel, *Math. Teacher* **61**, 527 (1968).  
[14] A.K. Bartlett, *Popular Astronomy* **12**, 649 (1904).  
[15] D.R. Abad, *Rev. Bras. Ensino Fís.* **43**, e20200482 (2021).  
[16] D. Halliday, R. Resnick and J. Walker, *Fundamentals of Physics Extended* (Wiley, Hoboken, 2013), 10 ed.  
[17] Error found in S.A.T. question, *The New York Times Archives*, May 25, 1982, available in: https://www.nytimes.com/1982/05/25/us/error-found-in-sat-question.html  
» https://www.nytimes.com/1982/05/25/us/error-found-in-sat-question.html  
[18] MINDYOURDECISIONS, *Why did everyone miss this SAT Math question?*, available in: https://www.youtube.com/watch?v=kN3AOMrnEUs  
» https://www.youtube.com/watch?v=kN3AOMrnEUs  
[19] J. Murtagh, *The SAT Problem That Everybody Got Wrong*, available in: https://www.scientificamerican.com/article/the-sat-problem-that-everybody-got-wrong/  
» https://www.scientificamerican.com/article/the-sat-problem-that-everybody-got-wrong/  
[20] VERITASIUM YOUTUBE CHANNEL, *The SAT Question Everyone Got Wrong*, available in: https://www.youtube.com/watch?v=FUHkTs-Ipfg  
» https://www.youtube.com/watch?v=FUHkTs-Ipfg  
[21] J.J. Uicker, G.R. Pennock and J.E. Shigley, *Theory of Machines and Mechanisms* (Oxford University Press, New York, 2003).  
[22] B. Paul, *Kinematics and Dynamics of Planar Machinery* (Prentice Hall, Englewood Cliffs, 1979).  
[23] J.A. Boyle, arXiv:1406.1736v1 (2014).  
[24] N.C. Rana and P.S. Joag, *Classical Mechanics* (Tata McGraw-Hill, Noida, 2001).  
[25] E.A. Whitman, *American Mathematical Monthly* **50**, 309 (1943).  
[26] S. Wagon, *Mathematica in Action* (Springer Science & Business Media, New York, 1999).  
[27] R. Courant and H. Robbins, *What Is Mathematics?: An Elementary Approach to Ideas and Methods* (Oxford University Press, Oxford, 1996), 2 ed.  
[28] S.T. Thornton and J.B. Marion, *Classical Dynamics of Particles and Systems* (Brooks/Cole, Pacific Grove, 2004), 5 ed.  
[29] VERITASIUM YOUTUBE CHANNEL, available in: https://www.youtube.com/@veritasium, accessed in: 15/10/2025.  
» https://www.youtube.com/@veritasium  
[30] 3BLUE1BROWN YOUTUBE CHANNEL, available in: https://www.youtube.com/c/3blue1brown, accessed in: 15/10/2025.  
» https://www.youtube.com/c/3blue1brown  
